# OvaNet AI — Day 5: Weakly-Supervised U-Net Training on Colab GPU

**Team:** TwoMuchCode (Madhumita & Sahina)  
**Track:** Vision Track (Madhumita)  
**Target:** Train a ResNet-18 U-Net on pseudo-masks from the classical follicle detector to automate ovary and follicle segmentation.

### Key Architecture Specifications:
- **Encoder:** ResNet-18 (ImageNet pretrained weights)
- **Input:** Grayscale ultrasound image `[256, 256]` in `[0, 1]`
- **Output:** 2 Channels (`Channel 0`: Ovary Sector ROI, `Channel 1`: Follicles)
- **Embedding ($E_{cnn}$):** 512-dim bottleneck embedding via Global Average Pooling
- **Loss:** Combined `BCEWithLogitsLoss` + `DiceLoss`

In [ ]:
# 1. GPU Check & Dependencies
!nvidia-smi
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless

In [ ]:
# 2. Environment Setup
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import cv2

# If running directly inside cloned repo on Colab:
if Path("/content/ovanet-ai").exists():
    %cd /content/ovanet-ai
    sys.path.insert(0, "/content/ovanet-ai")
else:
    sys.path.insert(0, str(Path.cwd()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active compute device: {device}")

In [ ]:
# 3. Unpack Pseudo-Masks (if zip provided)
import zipfile
zip_path = Path("data/pseudo_masks_train_val.zip")
if zip_path.exists():
    print(f"Unpacking {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("data/pseudo_masks")
    print("Unpacked pseudo-masks successfully.")
else:
    print("Using existing data/pseudo_masks directory.")

In [ ]:
# 4. Dataset & DataLoader
from ovanet.vision.dataset import OvaNetSegmentationDataset
from torch.utils.data import DataLoader

manifest_train = Path("data/pseudo_masks/manifest_train.csv")
manifest_val = Path("data/pseudo_masks/manifest_val.csv")
raw_root = Path("data/raw/pcos_primary/data")
mask_dir = Path("data/pseudo_masks")

train_df = pd.read_csv(manifest_train)
val_df = pd.read_csv(manifest_val)

train_ds = OvaNetSegmentationDataset(train_df, raw_root=raw_root, mask_dir=mask_dir, is_train=True)
val_ds = OvaNetSegmentationDataset(val_df, raw_root=raw_root, mask_dir=mask_dir, is_train=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

In [ ]:
# 5. Initialize Model, Loss, Optimizer
from ovanet.vision.unet import OvaNetUNet
from ovanet.vision.train_unet import CombinedBCEDiceLoss

model = OvaNetUNet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=1,
    classes=2
).to(device)

criterion = CombinedBCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
print("Model, loss function, and scheduler initialized.")

In [ ]:
# 6. Training Loop (15 Epochs on GPU)
from ovanet.vision.train_unet import train_one_epoch, evaluate

epochs = 15
best_val_dice = 0.0
weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)
best_weights_path = weights_dir / "unet_resnet18_best.pth"
history = []

for epoch in range(1, epochs + 1):
    train_loss, tr_ov, tr_fo = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, vl_ov, vl_fo = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    val_mean_dice = (vl_ov + vl_fo) / 2.0
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_ovary_dice": vl_ov,
        "val_follicle_dice": vl_fo,
        "val_mean_dice": val_mean_dice
    })
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Ovary Dice: {vl_ov:.3f} | Follicle Dice: {vl_fo:.3f} | Mean Dice: {val_mean_dice:.3f}")
    
    if val_mean_dice > best_val_dice:
        best_val_dice = val_mean_dice
        model.save_checkpoint(best_weights_path, epoch=epoch, metrics=history[-1])
        print(f"  --> [Saved Best Weights] {best_weights_path} (Val Mean Dice: {best_val_dice:.4f})")

print(f"\nTraining complete! Peak Validation Mean Dice: {best_val_dice:.4f}")

In [ ]:
# 7. Verify E_cnn Bottleneck Embedding (Contract C2)
sample = next(iter(val_loader))
with torch.no_grad():
    e_cnn = model.extract_encoder_embedding(sample["image"].to(device))
print(f"Extracted E_cnn batch shape: {e_cnn.shape}")
assert e_cnn.shape[1] == 512, f"Expected 512 dimensions, got {e_cnn.shape[1]}"
print("PASS: E_cnn dimension matches contract C2 (512-dim) perfectly!")

In [ ]:
# 8. Qualitative Prediction Visualizations
model.eval()
images = sample["image"].to(device)
masks = sample["mask"].numpy()
with torch.no_grad():
    preds = torch.sigmoid(model(images)).cpu().numpy()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(min(3, len(images))):
    img_show = sample["image"][i, 0].numpy()
    gt_ovary = masks[i, 0]
    gt_follicles = masks[i, 1]
    pred_ovary = (preds[i, 0] >= 0.5).astype(float)
    pred_follicles = (preds[i, 1] >= 0.5).astype(float)
    
    axes[i, 0].imshow(img_show, cmap="gray")
    axes[i, 0].set_title(f"Image: {sample['name'][i]}")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(img_show, cmap="gray")
    axes[i, 1].imshow(gt_ovary, alpha=0.3, cmap="Greens")
    axes[i, 1].imshow(gt_follicles, alpha=0.6, cmap="Reds")
    axes[i, 1].set_title("Ground Truth Pseudo-Mask")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(img_show, cmap="gray")
    axes[i, 2].imshow(pred_ovary, alpha=0.3, cmap="Blues")
    axes[i, 2].imshow(pred_follicles, alpha=0.6, cmap="hot")
    axes[i, 2].set_title("U-Net Prediction Overlay")
    axes[i, 2].axis("off")
    
    axes[i, 3].imshow(pred_follicles, cmap="magma")
    axes[i, 3].set_title("Predicted Follicle Regions")
    axes[i, 3].axis("off")

plt.tight_layout()
out_plot = Path("experiments/results/unet_sample_predictions.png")
out_plot.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_plot, dpi=150)
plt.show()
print(f"Sample predictions saved to {out_plot}")